# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_02 — AI_EXTRACT_PANDAS

## 1. Objetivos y requisitos
Extraer entidades de incidentes y valores estructurados sin Spark. Utilizar un runtime Fabric compatible, acceso al endpoint integrado y capacidad habilitada. Revisar los [requisitos oficiales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview) antes de instalar. Los datos son ficticios. Cada inferencia consume capacidad.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 3, Finished, Available, Finished, False)

## 2. Configuración
Modelo gpt-5-mini, concurrencia de laboratorio 10 y barra de progreso con estadísticas. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.extract con pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/extract)

```python
df_entities = df["text"].ai.extract("entity1", "entity2", "entity3")
```

| Parámetro | Tipo | Uso |
|---|---|---|
| labels | strings posicionales | Entidades a extraer |
| ExtractLabel | objeto | Definición alternativa de cada entidad |
| conf | Conf | Configuración opcional por llamada |
| column_type | string | `path` para rutas de archivos |

`ExtractLabel`: `label` identifica la entidad; `description` contextualiza; `type` selecciona el tipo JSON; `max_items` limita los resultados; las propiedades de JSON Schema permiten definir objetos y enumeraciones. `raw_col` permite conservar la respuesta sin procesar.

El retorno es un **DataFrame**, no una Series. Normalmente cada etiqueta contiene una lista; con `max_items=1` se espera un valor individual. Una ausencia de entidad puede producir un nulo. Revisar los tipos efectivos antes de transformar los resultados.


## 4. Dataset ficticio
Mismos cinco incidentes que en NB_02 PySpark.

In [3]:
data = [
    (
        1,
        "La solicitud fue reportada por Camila Torres de Contoso Chile, "
        "en Santiago. La aplicación afectada es Portal de Ventas. "
        "La prioridad es alta y hay 18 usuarios afectados."
    ),
    (
        2,
        "Andrés Molina, de Fabrikam, reporta desde Valparaíso una falla "
        "en el sistema ERP. El incidente tiene prioridad crítica y afecta "
        "a 42 usuarios."
    ),
    (
        3,
        "El contacto es Daniela Rojas de Northwind, ubicada en Concepción. "
        "Solicita soporte para Power BI. La prioridad es media y afecta "
        "a 6 usuarios."
    ),
    (
        4,
        "Javier Soto de Adventure Works, en Antofagasta, informa problemas "
        "con el acceso a Microsoft Teams. La prioridad es baja y afecta "
        "a 3 usuarios."
    ),
    (
        5,
        "María Pérez de Contoso Chile, desde Santiago, reporta que el "
        "sistema de inventario no permite registrar movimientos. "
        "La prioridad es alta y afecta a 11 usuarios."
    )
]

df = pd.DataFrame(data, columns=["id_incidente", "descripcion"])
display(df)

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0dacd230-da07-437b-8853-2a11cf908ae4)

## 5. Extracción básica
Cada llamada conserva el objeto original para recuperar estadísticas inmediatamente.

In [4]:
runs = []
def ejecutar(nombre, *labels):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc).isoformat()
    t0 = perf_counter()
    entidades = df["descripcion"].ai.extract(*labels)
    elapsed = perf_counter() - t0
    raw = entidades.ai.stats
    resultado = df.join(entidades)
    runs.append((nombre, run_id, started, elapsed, raw, resultado))
    return entidades, resultado

entidades_basicas, resultado_basico = ejecutar("basic", "contact_name", "organization", "city")
display(resultado_basico)

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 6, Finished, Available, Finished, False)

ai.extract: 100%|██████████| 5/5 [00:03<00:00,  1.36it/s, cached=0, in=1.93k, out=1.18k, CU-s=80.68]                          


SynapseWidget(Synapse.DataFrame, 9ab53485-da03-476a-8c53-848e830c717f)

## 6. Extracción estructurada
Se conserva el esquema del ejemplo PySpark: aplicación, prioridad y usuarios afectados. Esta segunda llamada genera consumo adicional.

In [7]:
incident_label = aifunc.ExtractLabel(
    label="incident_details",
    description=(
        "Extract the application name, incident priority and the number "
        "of affected users from the incident description. Do not infer "
        "values that are not present in the text."
    ),
    type="object",
    properties={
        "application": {
            "type": "string"
        },
        "priority": {
            "type": "string",
            "enum": ["critical", "high", "medium", "low"]
        },
        "affected_users": {
            "type": "integer"
        }
    },
    # "required" y "additionalProperties" no son argumentos admitidos en este entorno
    # para el constructor de ExtractLabel, por lo que se omiten.
    max_items=1
)

entidades_estructuradas, resultado = ejecutar("structured", incident_label)
display(resultado)

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 9, Finished, Available, Finished, False)

ai.extract: 100%|██████████| 5/5 [00:02<00:00,  1.79it/s, cached=0, in=1.57k, out=1.38k, CU-s=93.96]                           


SynapseWidget(Synapse.DataFrame, d5fb8357-2da4-4a5f-8171-c75e8e1c954d)

## 7. Validación
Los nulos no equivalen necesariamente a errores de inferencia. Comparar los usuarios afectados contra los valores conocidos; revisar manualmente discrepancias.

In [8]:
display(entidades_basicas.isna().sum())
print(entidades_estructuradas.dtypes)
def usuarios(v):
    if isinstance(v, list):
        v = v[0] if v else None
    return v.get("affected_users") if isinstance(v, dict) else None
validacion = pd.DataFrame({"id_incidente": df.id_incidente, "esperado": [18,42,6,3,11], "extraido": entidades_estructuradas["incident_details"].map(usuarios)})
validacion["coincide"] = validacion.esperado.eq(validacion.extraido)
display(validacion)

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d80e144-8c4e-4f66-9c4d-a885d99faad1)

incident_details    aifunc-result
dtype: object


SynapseWidget(Synapse.DataFrame, 580b0072-7256-4ae6-88d7-334bb6f8afd1)

## 8. Estadísticas y tokens
Consultar cada ejecución por separado. No reemplazar métricas ausentes por cero.

In [9]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")
for nombre, _, _, _, raw, _ in runs:
    print(nombre)
    display(raw)

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 11, Finished, Available, Finished, False)

basic


SynapseWidget(Synapse.DataFrame, e5404826-7080-48b3-b54f-69b5acdf33ea)

structured


SynapseWidget(Synapse.DataFrame, 2f7af84f-7a1d-40ff-a203-1ec52d1947db)

## 9. Estimación de CU-segundos
Tarifas de referencia de la serie para gpt-5-mini: entrada 8,40; caché 0,84; salida 67,23 CU-s por mil tokens. Consultar [facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing) para verificar vigencia.

$$CU_s=((I-C)8.40+C0.84+O67.23)/1000$$

El razonamiento es informativo; no se añade dos veces. La estimación no incluye otros consumos del runtime.

## 10. Auditoría individual y acumulada
Registrar ambas llamadas, tiempo y memoria aproximada de DataFrames (no memoria máxima del proceso).

In [10]:
records = []
for nombre, run_id, started, elapsed, raw, resultado_run in runs:
    stats = normalizar(raw)
    def metric(k):
        v = stats.get(k)
        return None if v is None or pd.isna(v) else int(v)
    I,C,O = [metric(k) for k in ("input_tokens","cached_tokens","output_tokens")]
    cu = None
    if (stats.get("model") in (MODEL_NAME,"gpt-5-mini-2025-08-07") and stats.get("client_type")=="fabric_llm_endpoint" and all(v is not None for v in (I,C,O)) and 0<=C<=I and O>=0):
        cu = ((I-C)*8.40+C*0.84+O*67.23)/1000
    audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_02_AI_EXTRACT_PANDAS", operation="ai.extract:"+nombre, model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=len(df), client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=int(df.memory_usage(deep=True).sum()), output_bytes=int(resultado_run.memory_usage(deep=True).sum()), concurrency=CONCURRENCY)
    for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
        audit[key]=metric(key)
    audit["execution_status"]="SUCCESS" if audit["num_successful"]==len(df) else "INCOMPLETE"
    records.append(audit)
audit_df=pd.DataFrame(records)
display(audit_df)
print("CU-s acumulados:", audit_df.estimated_cu_seconds.sum(min_count=len(audit_df)))

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 405f74b6-3cdf-48b5-9c84-3906539d1e17)

CU-s acumulados: 201.09702000000001


## 11. Persistencia opcional en Delta
Requiere Lakehouse adjunto, deltalake y pyarrow. Persiste en Files, no registra una tabla SQL. Deshabilitada por defecto.

In [11]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

StatementMeta(, 224fcf28-ba23-4885-bf52-0b2a0a179e43, 13, Finished, Available, Finished, False)

## 12. Multimodalidad (opcional)
Usar rutas autorizadas de archivos y `column_type="path"`. No ejecutar con datos sensibles sin aprobación.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = pd.Series(["/lakehouse/default/Files/ejemplo.pdf"])
    extraidos = archivos.ai.extract("organization", column_type="path")
    display(extraidos)
    display(extraidos.ai.stats)

## 13. Comparación y conclusiones
Pandas devuelve un DataFrame de entidades y realiza las llamadas de manera inmediata; PySpark incorpora columnas en un DataFrame distribuido. La extracción multimodal opcional no se incluye en la auditoría principal. Revisar resultados y costos reales en Fabric.

Referencias: [Extract](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/extract), [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration), [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).